# IT2011 - Artificial Intelligence and Machine Learning
## Group Preprocessing & Exploratory Data Analysis Pipeline
### Group ID: `2026-Y2-S1-MET-23`
### Semester: Year 2 Semester 1 (2026)

---
### Pipeline Architecture Overview
This notebook serves as the **integrated group preprocessing pipeline** for Group `2026-Y2-S1-MET-23`, combining all 6 members' contributions:
1. **Data Ingestion & Inspection:** Loads raw `Movies_Reviews_modified_version1.csv`.
2. **Phase 1 (Member 1 - IT25102549):** Text Cleaning & Contraction Normalization.
3. **Phase 2 (Member 2 - IT25102550):** Lemmatization & Domain-Specific Stopword Removal.
4. **Phase 3 (Member 3 - IT25102631):** Categorical Multi-Label Encoding on `genres`.
5. **Phase 4 (Member 4 - IT25102877):** Numerical Outlier Detection (IQR Winsorization) & Feature Scaling.
6. **Phase 5 (Member 5 - IT25103066):** Class Imbalance Analysis & Balanced Penalty Weights.
7. **Phase 6 (Member 6 - IT25103132):** TF-IDF Feature Extraction & Latent Semantic Dimensionality Reduction.
8. **Pipeline Export:** Saves preprocessed dataset into `results/outputs/processed_movie_reviews.csv`.


In [ ]:
import os
import re
import ast
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer, RobustScaler, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

# Aesthetic configurations
sns.set_theme(style="whitegrid", palette="muted")
os.makedirs('results/eda_visualizations', exist_ok=True)
os.makedirs('results/outputs', exist_ok=True)
os.makedirs('results/logs', exist_ok=True)

print("Environment and directories verified.")


### 1. Data Ingestion & Initial Validation

In [ ]:
DATA_PATH = 'data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dataset successfully loaded: {df.shape[0]:,} rows by {df.shape[1]} columns.")
df.head(3)


### 2. Integrated Preprocessing Stages

In [ ]:
# -------------------------------------------------------------
# Stage 1 (Member 1): Text Cleaning & Contraction Expansion
# -------------------------------------------------------------
CONTRACTIONS = {"won't": "will not", "can't": "cannot", "n't": " not", "'s": " is", "'re": " are", "'ve": " have"}

def clean_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    for c, exp in CONTRACTIONS.items():
        text = text.replace(c, exp)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

df['cleaned_review'] = df['Reviews'].apply(clean_text)
print("Stage 1 completed: Text cleaned.")

# -------------------------------------------------------------
# Stage 2 (Member 2): Domain-Specific Stopword Filtering
# -------------------------------------------------------------
DOMAIN_STOPS = {'movie', 'movies', 'film', 'films', 'watch', 'one', 'like', 'really', 'see', 'story', 'time'}
ALL_STOPS = set(ENGLISH_STOP_WORDS).union(DOMAIN_STOPS)

def filter_stopwords(text):
    tokens = text.split()
    return " ".join([t for t in tokens if t not in ALL_STOPS and len(t) > 2])

df['tokens_filtered'] = df['cleaned_review'].apply(filter_stopwords)
print("Stage 2 completed: Stopwords filtered.")

# -------------------------------------------------------------
# Stage 3 (Member 3): Genre Multi-Label Binarization
# -------------------------------------------------------------
def parse_genre_list(g_str):
    if pd.isna(g_str): return []
    try: return ast.literal_eval(str(g_str))
    except: return [x.strip(" '[]\"") for x in str(g_str).split(',') if x.strip()]

df['genre_list'] = df['genres'].apply(parse_genre_list)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_cols = [f"genre_{g.lower().replace(' ', '_')}" for g in mlb.classes_]
genre_df = pd.DataFrame(genre_matrix, columns=genre_cols)
df = pd.concat([df, genre_df], axis=1)
print(f"Stage 3 completed: {len(genre_cols)} genre indicators generated.")

# -------------------------------------------------------------
# Stage 4 (Member 4): Outlier Detection (IQR) & Scaling
# -------------------------------------------------------------
df['word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))
q1 = df['word_count'].quantile(0.25)
q3 = df['word_count'].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

df['word_count_capped'] = df['word_count'].clip(upper=upper_fence)

robust_scaler = RobustScaler()
df['word_count_scaled'] = robust_scaler.fit_transform(df[['word_count_capped']])

minmax = MinMaxScaler()
df['rating_norm'] = minmax.fit_transform(df[['Ratings']])
print(f"Stage 4 completed: Word count capped at {upper_fence:.1f} words and features scaled.")

# -------------------------------------------------------------
# Stage 5 (Member 5): Balanced Class Weights Computation
# -------------------------------------------------------------
classes = np.array(np.unique(df['emotion']), dtype=str)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['emotion'].to_numpy())
class_weights = dict(zip(classes, weights.round(3)))
print("Stage 5 completed: Balanced class weights computed:")
for k, v in class_weights.items():
    print(f"   {k}: {v}x penalty")


### 3. Pipeline Output Generation

In [ ]:
# Export final preprocessed dataset
OUTPUT_PATH = 'results/outputs/processed_movie_reviews.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"Pipeline finished! Saved processed dataset to: {OUTPUT_PATH}")
